In [ ]:
!pip install pandas
!pip install numpy
!pip install statsbombpy
%pip install mplsoccer

You should consider upgrading via the '/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import pandas as pd
import numpy as np 
from mplsoccer import Pitch
from statsbombpy import sb

In [ ]:
comps = sb.competitions()
comps
matches = sb.matches(competition_id=55, season_id=282) #Euro 2024 Data

events = sb.events(match_id=matches.match_id.iloc[0])
print(events.columns.tolist())

['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'block_deflection', 'block_save_block', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_outcome', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_penalty', 'foul_won_advantage', 'foul_won_defensive', 'foul_won_penalty', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_punched_out', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'injury_stoppage_in_chain', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_length', 'pass_outcome', 'pass_outswinging', 'pass_recipient', 'pass_recipient_

In [17]:
throw_ins = events[events['pass_type'] == 'Throw-in']
print(f"Throw-ins in this match: {len(throw_ins)}")
throw_ins[['location', 'pass_end_location', 'team', 'minute', 'possession']].head(10)

Throw-ins in this match: 25


,location,pass_end_location,team,minute,possession
121,"[110.6, 0.1]","[107.6, 6.6]",England,11,15
132,"[106.2, 0.1]","[83.2, 4.4]",England,12,18
149,"[39.4, 0.1]","[91.4, 8.8]",England,19,21
156,"[64.8, 0.1]","[37.7, 19.2]",Netherlands,19,23
163,"[51.5, 80.0]","[32.0, 63.4]",England,20,24
185,"[34.7, 0.1]","[29.5, 12.9]",Netherlands,21,26
196,"[103.2, 80.0]","[88.8, 71.0]",England,22,28
246,"[85.1, 80.0]","[80.5, 74.1]",Netherlands,26,32
348,"[18.3, 80.0]","[47.8, 66.3]",Netherlands,36,41
409,"[111.5, 80.0]","[105.7, 65.2]",England,40,46


In [19]:
sample = throw_ins.iloc[0]
print("Throw location:", sample['location'])
print("Pass end location:", sample['pass_end_location'])
print("Team:", sample['team'])
print("Minute:", sample['minute'])

Throw location: [110.6, 0.1]
Pass end location: [107.6, 6.6]
Team: England
Minute: 11


In [20]:
out_events = events[events['out'] == True]
print(f"Ball exit events: {len(out_events)}")
out_events[['type', 'location', 'pass_end_location', 'carry_end_location']].head(10)

Ball exit events: 21


,type,location,pass_end_location,carry_end_location
3176,Miscontrol,"[61.2, 15.7]",NaN,NaN
3181,Miscontrol,"[112.1, 12.8]",NaN,NaN
3182,Miscontrol,"[69.8, 4.6]",NaN,NaN
3183,Miscontrol,"[64.8, 74.5]",NaN,NaN
3184,Miscontrol,"[90.6, 77.5]",NaN,NaN
3186,Miscontrol,"[97.2, 76.9]",NaN,NaN
3187,Miscontrol,"[90.5, 4.0]",NaN,NaN
3314,Block,"[16.4, 9.6]",NaN,NaN
3317,Block,"[15.8, 34.9]",NaN,NaN
3318,Block,"[17.6, 29.2]",NaN,NaN


In [21]:
# sort events
events_sorted = events.sort_values('index').reset_index(drop=True)

# get the event immediately before each throw-in
events_sorted['next_pass_type'] = events_sorted['pass_type'].shift(-1)
ball_exits = events_sorted[events_sorted['next_pass_type'] == 'Throw-in'].copy()

# check what type those exit events are
print(ball_exits['type'].value_counts())

# check their locations
ball_exits[['type', 'location', 'pass_end_location', 'carry_end_location', 'out']].head(10)

type
Block            6
Miscontrol       5
Ball Receipt*    4
Duel             3
Clearance        2
Goal Keeper      2
Interception     1
Pass             1
Bad Behaviour    1
Name: count, dtype: int64


,type,location,pass_end_location,carry_end_location,out
405,Clearance,"[4.3, 46.0]",NaN,NaN,True
446,Goal Keeper,"[1.7, 40.4]",NaN,NaN,NaN
509,Ball Receipt*,"[80.4, 71.5]",NaN,NaN,NaN
531,Interception,"[46.2, 75.4]",NaN,NaN,NaN
556,Miscontrol,"[61.2, 15.7]",NaN,NaN,True
627,Duel,"[87.6, 72.6]",NaN,NaN,NaN
667,Clearance,"[3.3, 38.9]",NaN,NaN,True
838,Block,"[16.4, 9.6]",NaN,NaN,True
1163,Ball Receipt*,"[101.6, 12.7]",NaN,NaN,NaN
1348,Goal Keeper,"[2.7, 38.1]",NaN,NaN,NaN
